# 22 — Modelos semanales independientes por horizonte

Compara el sistema actual agrupado (H1–H2/H3–H5) con cinco ajustes H1–H5. El tuning se hace sólo en una ventana temporal previa a 2026 y se congela antes de evaluar semanas nuevas. Se prueba una cuadrícula pequeña y legible de árboles; la decisión se documenta por WAPE de validación. El clima se incorpora sólo si tiene cobertura suficiente en train y test; si no, esa ejecución cambia automáticamente a la versión sin clima.

In [1]:
# Datos comunes y evaluación sin fuga temporal
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from threadpoolctl import threadpool_limits

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Datos_analiticos_proyecto").is_dir())
DATA = ROOT / "Datos_analiticos_proyecto"
OUT = DATA / "experimentos_actualizacion_20261020"
OUT.mkdir(parents=True, exist_ok=True)
DEST = DATA / "modelo_con_proyecciones_teoricas"
K = ["finca", "producto", "color", "variedad"]
CASE = K + ["origen", "semana_objetivo", "horizonte"]

# Features calculadas al origen. y se sustituye por la producción 2026 recién limpia,
# pero las entradas históricas del caso quedan congeladas como estaban al emitir.
base = pd.read_parquet(DEST / "base_analitica_proyecciones.parquet")
real26 = pd.read_parquet(DATA / "produccion_real_2026_actualizada.parquet")
real26 = real26.loc[real26.finca.eq("GAITANA") & real26.llave_completa].copy()
real_semanal = real26.groupby(K + ["semana_inicio"], as_index=False).agg(
    y_nuevo=("tallos", "sum"), ultimo_dia_real=("fecha_corte", "max"))
ultima_fecha = real26.fecha_corte.max()
ultima_semana = ultima_fecha - pd.Timedelta(days=ultima_fecha.weekday())
ultima_semana_completa = ultima_semana if ultima_fecha.weekday() == 6 else ultima_semana - pd.Timedelta(days=7)

casos = base.loc[base.finca.eq("GAITANA")].copy()
casos = casos.merge(real_semanal, left_on=K + ["semana_objetivo"],
                    right_on=K + ["semana_inicio"], how="left", validate="many_to_one")
casos["y_eval"] = casos.y_nuevo.where(casos.y_nuevo.notna(), casos.y)
casos["admisible_actual"] = casos.media_4.notna() & casos.observaciones_previas.ge(26)
casos["horizonte"] = casos.horizonte.astype(int)
model_data = casos.loc[casos.y_eval.notna() & casos.admisible_actual].copy()

# Mismo cruce nominal estricto del benchmark oficial: suma las líneas de la misma
# serie/origen/objetivo/horizonte y excluye cantidades incompletas y envíos posteriores.
e = pd.read_parquet(DATA / "estimados_semanales_limpio.parquet")
e = e.loc[e.finca.eq("GAITANA") & e.en_h1_h5 & ~e.es_ajustado
          & e.origen_solicitado.dt.year.eq(2026) & e.semana_inicio.dt.year.eq(2026)].copy()
e = e.rename(columns={"origen_solicitado": "origen", "semana_inicio": "semana_objetivo"})
engineer = e.groupby(CASE, dropna=False).agg(
    ingeniero=("estimado", "sum"), lineas=("estimado", "size"),
    cantidades=("estimado", "count"), archivos=("archivo_origen", "nunique"),
    posterior=("archivo_es_posterior", "max")).reset_index()
engineer["ingeniero_completo"] = engineer.lineas.eq(engineer.cantidades) & engineer.archivos.eq(1) & ~engineer.posterior.fillna(False)
casos = casos.merge(engineer, on=CASE, how="inner", validate="one_to_one")
casos = casos.loc[casos.ingeniero_completo & casos.y_eval.notna() & casos.admisible_actual].copy()
casos = casos.loc[casos.semana_objetivo.gt(pd.Timestamp("2026-08-31"))
                  & casos.semana_objetivo.le(ultima_semana_completa)].copy()
print("Última fecha real:", ultima_fecha.date(), "| última semana completa:", ultima_semana_completa.date())
print("Casos nuevos comparables tras cruce estricto:", len(casos))
print("Semanas objetivo:", sorted(casos.semana_objetivo.dt.strftime("%Y-%m-%d").unique()))
print("Clima disponible hasta:", pd.to_datetime(pd.read_parquet(DATA / "clima_limpio.parquet").fecha).max().date())


Última fecha real: 2026-10-03 | última semana completa: 2026-09-21
Casos nuevos comparables tras cruce estricto: 1226
Semanas objetivo: ['2026-09-07', '2026-09-14', '2026-09-21']
Clima disponible hasta: 2026-09-21


In [2]:
CATS = ["producto", "color", "variedad"]
MEMORY = ["produccion_lag1", "produccion_lag2", "produccion_lag3", "media_4", "media_8",
          "desv_4", "tendencia_4_8", "cambio_reciente", "produccion_52"]
CLIMATE = ["temperatura_semana_lag1", "temperatura_semana_media4",
           "humedad_semana_lag1", "humedad_semana_media4",
           "radiacion_semana_lag1", "radiacion_semana_media4"]
THEORY = ["teoria_modelo", "antiguedad_curva", "desviacion_reciente_teoria",
          "teoria_previa_1", "teoria_previa_2", "revision_teoria"]
BASE_X = CATS + ["horizonte", "semana_seno", "semana_coseno"] + MEMORY

def metricas(g, p):
    y = np.asarray(g.y_eval, dtype=float); p = np.asarray(p, dtype=float)
    den = y.sum()
    return {"n": len(y), "MAE_tallos": float(np.mean(np.abs(y-p))),
            "WAPE_pct": float(100*np.abs(y-p).sum()/den) if den else np.nan,
            "sesgo_pct": float(100*(p-y).sum()/den) if den else np.nan}

def elegir_clima(train, test, habilitar=True, minimo=0.80):
    """Usa clima sólo si existe cobertura suficiente en el ajuste evaluado."""
    if not habilitar or not all(c in train and c in test for c in CLIMATE):
        return [], 0.0, "sin clima: columnas no disponibles"
    cov_train = float(train[CLIMATE].notna().mean().mean())
    latest_origin = test.origen.max()
    current = test.loc[test.origen.eq(latest_origin), CLIMATE]
    cov_current = float(current.notna().mean().mean()) if len(current) else 0.0
    cobertura = min(cov_train, cov_current)
    if cobertura < minimo:
        return [], cobertura, f"sin clima: cobertura {cobertura:.1%} < {minimo:.0%}"
    return CLIMATE, cobertura, f"con clima: cobertura {cobertura:.1%}"

def crear_modelo(cols, params=None):
    params = params or {}
    cats = [c for c in CATS if c in cols]
    nums = [c for c in cols if c not in cats]
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), cats),
        ("num", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), nums),
    ])
    reg = HistGradientBoostingRegressor(
        learning_rate=params.get("learning_rate", .08), max_iter=params.get("max_iter", 80),
        max_leaf_nodes=params.get("max_leaf_nodes", 15), min_samples_leaf=40,
        l2_regularization=1.0, early_stopping=False, random_state=42)
    return make_pipeline(prep, reg)

def pronosticar_ruta(train, test, cols, params=None):
    """Ajusta la media4 y, con soporte suficiente, la teoría completa por separado."""
    train = train.loc[train.y_eval.notna() & train.media_4.notna()].copy()
    pred = pd.Series(index=test.index, dtype=float)
    if train.empty or test.empty: return pred
    if len(train) > 5000: train = train.sample(5000, random_state=42)
    if len(train) > 5000: train = train.sample(5000, random_state=42)
    with threadpool_limits(limits=2):
        fallback = crear_modelo(cols, params)
        fallback.fit(train[cols], train.y_eval-train.media_4)
        pred.loc[test.index] = np.maximum(0, test.media_4 + fallback.predict(test[cols]))
        tr = train.loc[train.curva_completa.fillna(False) & train.teoria_modelo.notna()]
        te = test.loc[test.curva_completa.fillna(False) & test.teoria_modelo.notna()]
        if len(tr) >= 40 and len(te):
            use = [c for c in cols + THEORY if c in train.columns]
            teoria = crear_modelo(use, params)
            teoria.fit(tr[use], tr.y_eval-tr.teoria_modelo)
            pred.loc[te.index] = np.maximum(0, te.teoria_modelo + teoria.predict(te[use]))
    return pred

def fuera_producto(df, patron):
    return df.producto.isin(str(patron).split("|"))

def evaluar_origenes(data, grupos="agrupado", excluir_entrenamiento=None, clima="auto", params=None, train_pool=None):
    """Reentrena sólo con semanas reales cerradas antes de cada origen evaluado."""
    train_pool = model_data if train_pool is None else train_pool
    salidas=[]; controles=[]
    for origen, test in data.groupby("origen", sort=True):
        test = test.copy()
        train = train_pool.loc[train_pool.semana_objetivo + pd.Timedelta(days=7) <= origen].copy()
        if excluir_entrenamiento:
            train = train.loc[~fuera_producto(train, excluir_entrenamiento)]
        if train.empty: continue
        climat, cov, modo = elegir_clima(train, test, habilitar=(clima != "sin"))
        x = BASE_X + climat
        test["prediccion"] = np.nan
        grp = test.horizonte if grupos == "horizonte" else np.where(test.horizonte <= 2, "H1_H2", "H3_H5")
        test["_grp"] = grp
        train["_grp"] = train.horizonte if grupos == "horizonte" else np.where(train.horizonte <= 2, "H1_H2", "H3_H5")
        for gval, te in test.groupby("_grp"):
            tr = train.loc[train._grp.eq(gval)]
            if tr.empty: continue
            teorio = x + THEORY
            te["prediccion"] = pronosticar_ruta(tr, te, x, params)
            test.loc[te.index, "prediccion"] = te.prediccion
            if test.loc[te.index, "prediccion"].notna().any():
                controles.append({"origen": origen, "grupo": str(gval), "train_n": len(tr),
                                  "test_n": len(te), "clima": modo, "cobertura_clima": cov})
        test.drop(columns="_grp", inplace=True)
        salidas.append(test)
    if not salidas: return pd.DataFrame(), pd.DataFrame(controles)
    return pd.concat(salidas).dropna(subset=["prediccion"]), pd.DataFrame(controles)

def resumen_comparacion(pred):
    rows=[]
    for nombre,col in [("Modelo", "prediccion"),("Ingeniero","ingeniero"),("Media 4 semanas","media_4"),("Teoría aplicable","teoria_modelo")]:
        g=pred.loc[pred[col].notna()].copy()
        if len(g): rows.append({"metodo":nombre,**metricas(g,g[col])})
    return pd.DataFrame(rows)


In [3]:
# Tuning anterior a 2026; luego evaluación nueva sin volver a mirar el tuning
valid = model_data.loc[model_data.origen.between("2025-11-03","2025-12-07")].copy()
test = casos.copy()
grilla = [{"max_iter":40,"max_leaf_nodes":15},{"max_iter":80,"max_leaf_nodes":15}]
seleccion=[]
for h in range(1,6):
    v=valid.loc[valid.horizonte.eq(h)].copy()
    tr=model_data.loc[model_data.horizonte.eq(h)&(model_data.semana_objetivo+pd.Timedelta(days=7)<=pd.Timestamp("2025-11-03"))].copy()
    climat,cov,modo=elegir_clima(tr,v,habilitar=True)
    x=BASE_X+climat
    if len(tr)<80 or v.empty: continue
    for p in grilla:
        pv=pronosticar_ruta(tr,v,x,p)
        z=v.loc[pv.notna()].copy();z["p"]=pv.loc[z.index]
        if len(z): seleccion.append({"horizonte":h,**p,"valid_n":len(z),"valid_WAPE_pct":100*(z.y_eval-z.p).abs().sum()/z.y_eval.sum(),"clima":modo,"cobertura_clima":cov})
tuning=pd.DataFrame(seleccion)
if tuning.empty: raise ValueError("No hubo soporte suficiente para tuning temporal por horizonte.")
mejores=tuning.sort_values(["horizonte","valid_WAPE_pct","max_iter","max_leaf_nodes"]).groupby("horizonte").first().reset_index()
display(tuning.sort_values(["horizonte","valid_WAPE_pct"]));display(mejores)

# Predicciones para mismas semanas/casos: agrupado actual versus un ajuste independiente por H.
origin_rows=[]
for h in range(1,6):
    ph=mejores.loc[mejores.horizonte.eq(h)]
    params=ph.iloc[0][["max_iter","max_leaf_nodes"]].to_dict() if len(ph) else None
    p=casos.loc[casos.horizonte.eq(h)].copy()
    z,_=evaluar_origenes(p,grupos="horizonte",clima="auto",params=params,train_pool=model_data)
    if len(z): z["experimento"]="Modelo H"+str(h);origin_rows.append(z)
general,_=evaluar_origenes(casos,grupos="agrupado",clima="auto",train_pool=model_data)
if len(general): general["experimento"]="Agrupado H1-H2/H3-H5";origin_rows.append(general)
pred=pd.concat(origin_rows,ignore_index=True)
rows=[]
for nombre,g in pred.groupby("experimento"):
    rows.append({"experimento":nombre,"metodo":"Modelo",**metricas(g,g.prediccion)})
    rows.append({"experimento":nombre,"metodo":"Ingeniero",**metricas(g,g.ingeniero)})
res=pd.DataFrame(rows)
rows_h=[]
for (nombre,h),g in pred.groupby(["experimento","horizonte"]):
    rows_h.append({"experimento":nombre,"horizonte":h,"metodo":"Modelo",**metricas(g,g.prediccion)})
    rows_h.append({"experimento":nombre,"horizonte":h,"metodo":"Ingeniero",**metricas(g,g.ingeniero)})
por_h=pd.DataFrame(rows_h)
display(res);display(por_h)
OUT.mkdir(exist_ok=True)
tuning.to_csv(OUT/"tuning_horizontes.csv",index=False);res.to_csv(OUT/"resultados_modelos_horizonte.csv",index=False);por_h.to_csv(OUT/"horizonte_detalle.csv",index=False)
pred.to_parquet(OUT/"predicciones_modelos_horizonte.parquet",index=False)


C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\entorno_tesis\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] El sistema no puede encontrar el archivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\entorno_tesis\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
  File "C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\entorno_tesis\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".spli

,horizonte,max_iter,max_leaf_nodes,valid_n,valid_WAPE_pct,clima,cobertura_clima
1,1,80,15,603,14.782592,con clima: cobertura 100.0%,1.0
0,1,40,15,603,14.837322,con clima: cobertura 100.0%,1.0
3,2,80,15,602,21.063097,con clima: cobertura 100.0%,1.0
2,2,40,15,602,21.295874,con clima: cobertura 100.0%,1.0
4,3,40,15,595,25.151698,con clima: cobertura 100.0%,1.0
5,3,80,15,595,25.588302,con clima: cobertura 100.0%,1.0
6,4,40,15,584,28.006226,con clima: cobertura 100.0%,1.0
7,4,80,15,584,28.512539,con clima: cobertura 100.0%,1.0
9,5,80,15,579,28.075305,con clima: cobertura 100.0%,1.0
8,5,40,15,579,29.356699,con clima: cobertura 100.0%,1.0


,horizonte,max_iter,max_leaf_nodes,valid_n,valid_WAPE_pct,clima,cobertura_clima
0,1,80,15,603,14.782592,con clima: cobertura 100.0%,1.0
1,2,80,15,602,21.063097,con clima: cobertura 100.0%,1.0
2,3,40,15,595,25.151698,con clima: cobertura 100.0%,1.0
3,4,40,15,584,28.006226,con clima: cobertura 100.0%,1.0
4,5,80,15,579,28.075305,con clima: cobertura 100.0%,1.0


,experimento,metodo,n,MAE_tallos,WAPE_pct,sesgo_pct
0,Agrupado H1-H2/H3-H5,Modelo,1226,2314.541203,18.256500,-5.477878
1,Agrupado H1-H2/H3-H5,Ingeniero,1226,2072.071778,16.343964,-7.507109
2,Modelo H1,Modelo,103,1588.000569,12.939028,3.745207
3,Modelo H1,Ingeniero,103,1869.281553,15.230905,-7.793917
4,Modelo H2,Modelo,205,1827.538990,14.956021,-0.015055
5,Modelo H2,Ingeniero,205,1830.541463,14.980593,-5.803677
6,Modelo H3,Modelo,306,2151.275732,16.773135,-9.733610
7,Modelo H3,Ingeniero,306,2118.029412,16.513919,-7.990249
8,Modelo H4,Modelo,306,2497.363201,19.471521,-7.035950
9,Modelo H4,Ingeniero,306,2154.049020,16.794758,-7.668694


,experimento,horizonte,metodo,n,MAE_tallos,WAPE_pct,sesgo_pct
0,Agrupado H1-H2/H3-H5,1,Modelo,103,1506.950594,12.278632,2.746177
1,Agrupado H1-H2/H3-H5,1,Ingeniero,103,1869.281553,15.230905,-7.793917
2,Agrupado H1-H2/H3-H5,2,Modelo,205,1840.841319,15.064884,1.969701
3,Agrupado H1-H2/H3-H5,2,Ingeniero,205,1830.541463,14.980593,-5.803677
4,Agrupado H1-H2/H3-H5,3,Modelo,306,2183.429658,17.023834,-9.012546
5,Agrupado H1-H2/H3-H5,3,Ingeniero,306,2118.029412,16.513919,-7.990249
6,Agrupado H1-H2/H3-H5,4,Modelo,306,2520.423088,19.651315,-6.679711
7,Agrupado H1-H2/H3-H5,4,Ingeniero,306,2154.049020,16.794758,-7.668694
8,Agrupado H1-H2/H3-H5,5,Modelo,306,2828.954880,22.056886,-8.143829
9,Agrupado H1-H2/H3-H5,5,Ingeniero,306,2174.205882,16.951918,-7.857245
